# Hybrid Search (Enterprise AI Pattern)

Hybrid Search is one of the **most frequently asked Advanced RAG topics** because enterprise applications rarely rely on **only vector search** or **only keyword search**.

Interviewers commonly ask:

- What is Hybrid Search?
- Why not use only vector search?
- BM25 vs Vector Search?
- How does Hybrid Search improve RAG?
- Explain Hybrid Search architecture.

---

# 1. What is Hybrid Search?

## Definition

**Hybrid Search** combines **Keyword Search (Lexical Search)** and **Vector Search (Semantic Search)** to retrieve more relevant documents.

Instead of relying on only one retrieval technique, it merges the results from both and optionally reranks them.

---

## Interview Answer

> Hybrid Search combines lexical search (BM25/keyword search) with semantic vector search to improve retrieval accuracy. Keyword search captures exact matches, while vector search captures semantic meaning. Combining both provides better recall and precision for enterprise RAG systems.

---

# 2. Why Do We Need Hybrid Search?

Suppose your document contains:

```text
Annual Leave Policy
Employees receive 20 annual leave days.
```

User asks

```text
PTO Policy
```

---

### Keyword Search

Looks for

```text
PTO
```

Document contains

```text
Annual Leave
```

Result

```text
No Match
```

---

### Vector Search

Embeddings understand

```text
PTO

≈

Paid Time Off

≈

Annual Leave
```

Result

```text
Correct Document
```

---

Now another example

Document

```text
Employee ID : EMP001
```

User asks

```text
EMP001
```

---

Vector Search

❌ May fail because IDs have little semantic meaning.

---

Keyword Search

✅ Exact match.

---

Therefore

```text
Keyword

+

Vector

=

Hybrid Search
```

---

# 3. Keyword Search vs Vector Search

| Keyword Search | Vector Search |
|---------------|---------------|
| Exact words | Semantic meaning |
| Uses BM25 | Uses embeddings |
| Good for IDs | Good for natural language |
| Fast | Computationally heavier |
| Doesn't understand synonyms | Understands synonyms |

---

# 4. Hybrid Search Architecture

```text
                    User Question
                           │
                           ▼
                     Query Processing
                           │
             ┌─────────────┴─────────────┐
             ▼                           ▼
      Keyword Search               Vector Search
        (BM25)                   (Embeddings)
             │                           │
             ▼                           ▼
     Matching Docs              Similar Documents
             └─────────────┬─────────────┘
                           ▼
                    Merge Results
                           ▼
                       Reranker
                           ▼
                    Top-K Documents
                           ▼
                  AWS Bedrock /
                  Azure OpenAI
                           ▼
                       Response
```

---

# AWS + Azure Components

| Layer | AWS | Azure |
|--------|------|--------|
| LLM | Bedrock | Azure OpenAI |
| Search | OpenSearch | Azure AI Search |
| Embeddings | Titan | text-embedding-3-large |
| Storage | S3 | Blob Storage |

---

# 5. Enterprise Flow

```text
User

↓

FastAPI

↓

LangGraph

↓

Hybrid Retriever

↓

BM25 Search

+

Vector Search

↓

Merge

↓

Reranker

↓

Top 5 Chunks

↓

Bedrock

↓

Response
```

---

# 6. Example

Document

```text
Annual Leave Policy
```

Question

```text
PTO Policy
```

---

Keyword Score

```text
0
```

---

Vector Score

```text
0.93
```

---

Hybrid

```text
0 + 0.93

↓

Retrieved
```

---

Second Example

Document

```text
Invoice Number

INV-1001
```

Question

```text
INV-1001
```

---

Keyword

```text
1.0
```

---

Vector

```text
0.45
```

---

Hybrid

```text
1.45

↓

Retrieved
```

---

# 7. Simple LangChain Example (Concept)

```python
# STEP 1 : Create BM25 Retriever
from langchain_community.retrievers import BM25Retriever

bm25 = BM25Retriever.from_documents(documents)
bm25.k = 3


# STEP 2 : Create Vector Retriever
vector_retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)


# STEP 3 : Retrieve Results
keyword_docs = bm25.invoke(question)

vector_docs = vector_retriever.invoke(question)


# STEP 4 : Merge Results
all_docs = keyword_docs + vector_docs


# STEP 5 : Remove Duplicates
unique_docs = list({doc.page_content: doc for doc in all_docs}.values())
```

> **Note:** This demonstrates the idea. Production systems typically use built-in hybrid capabilities (e.g., OpenSearch or Azure AI Search) and reranking instead of simply concatenating results.

---

# 8. Production Architecture

```text
User

↓

FastAPI

↓

JWT

↓

LangGraph

↓

Hybrid Search

├── BM25

├── Vector Search

↓

Merge

↓

Reranker

↓

Context Compression

↓

Bedrock

↓

Redis

↓

Response
```

---

# 9. Scoring

Production systems often compute a combined score.

Example

```text
Final Score

=

0.4 × Keyword Score

+

0.6 × Vector Score
```

The weights depend on the application and are tuned based on evaluation results.

---

# 10. Advantages

✅ Higher accuracy

✅ Better recall

✅ Handles IDs and natural language

✅ Fewer missed documents

✅ Better enterprise search

---

# 11. Disadvantages

❌ Slightly slower

❌ More infrastructure

❌ More tuning

❌ More storage

---

# 12. Best Practices

✅ Use BM25 + Vector Search

✅ Add reranking

✅ Use metadata filters

✅ Tune Top-K

✅ Evaluate retrieval quality using metrics such as Recall@K and MRR

---

# 13. Common Mistakes

❌ Only vector search

❌ Only keyword search

❌ No reranker

❌ Large Top-K

❌ No metadata filters

---

# 14. Hybrid Search vs Vector Search

| Vector Search | Hybrid Search |
|---------------|---------------|
| Semantic only | Semantic + Keyword |
| Misses IDs | Finds IDs |
| Misses exact matches | Better precision |
| Good | Better |

---

# 15. Real Enterprise Example

### HR Assistant

Question

```text
PTO policy
```

Document

```text
Annual Leave Policy
```

Vector Search finds it because it understands the meaning.

---

Question

```text
Employee ID EMP123
```

Keyword Search retrieves it through an exact match.

---

Hybrid Search

```text
Question

↓

Keyword Search

+

Vector Search

↓

Merge

↓

Rerank

↓

LLM
```

---

# 16. Interview Questions

### Q1. Why Hybrid Search?

To combine the strengths of keyword search and semantic search, improving retrieval quality.

---

### Q2. Why not only vector search?

Vector search can struggle with exact values such as employee IDs, invoice numbers, error codes, product SKUs, and abbreviations.

---

### Q3. Why not only keyword search?

Keyword search cannot understand synonyms, semantic similarity, or natural language intent.

---

### Q4. Does OpenSearch support Hybrid Search?

Yes.

Amazon OpenSearch supports combining lexical (BM25) and vector search.

Azure AI Search also supports hybrid search.

---

### Q5. Where is Hybrid Search used?

- HR Copilot
- Banking Assistant
- Healthcare Assistant
- Legal Search
- Enterprise Search
- Customer Support AI

---

# 17. EPAM Senior Answer (3 Minutes)

> "Hybrid Search combines lexical search using BM25 with semantic vector search to improve retrieval accuracy in RAG systems. BM25 is excellent for exact matches such as employee IDs, invoice numbers, and product codes, while vector search retrieves semantically similar documents even when different terminology is used. In an enterprise AI application, a user's query is processed by both retrieval mechanisms in parallel. The results are merged and often reranked before the most relevant chunks are sent to Amazon Bedrock or Azure OpenAI for response generation. On AWS, I typically implement this using Amazon OpenSearch with Titan Embeddings, while on Azure I use Azure AI Search with Azure OpenAI Embeddings. Hybrid Search significantly improves recall and precision, making it the preferred retrieval strategy for production RAG systems."
\